# Demo: Demand Propagation Workflow

This notebook walks through the `demand_propagation` package:

1. **`state.py`** — the closed-form state variables: the sanitizer-exposure
   index and the lotion-inventory stock.
2. **`synthetic_panel.py`** — a synthetic household-week panel shaped like the
   real (restricted) research data, generated from those state variables.
3. **`simulate.py`** — a forward simulation of how lotion purchases respond,
   week by week, to an extra sanitizer purchase, summarized by two multipliers.

No data files are needed and no real data is used.

**Before running:** follow the *Installation* steps in the repository
`README.md`, start Jupyter from the activated environment (`make serve`), and
choose **Kernel → Restart Kernel and Run All Cells**.

In [ ]:
from demand_propagation.simulate import response_multipliers, simulate_response
from demand_propagation.state import bin_inventory, exposure_index, inventory_transition
from demand_propagation.synthetic_panel import generate_synthetic_panel

## 1. Exposure index

The exposure index summarizes a household's sanitizer purchases over the last
nine weeks as a weighted average with geometrically decaying weights:

$$
E_t = \frac{\sum_{k=1}^{9} \eta^{\,k-1}\, s_{t-k}}{\sum_{k=1}^{9} \eta^{\,k-1}},
\qquad 0 < \eta \le 1,
$$

where $s_{t-k} \in \{0, 1\}$ is whether sanitizer was bought $k$ weeks ago.
The input list is ordered from the most recent week (lag 1) to the oldest
(lag 9).

Below, the household bought sanitizer **only in the most recent week**.
A smaller $\eta$ puts more weight on recent weeks, so the index is larger;
$\eta = 1$ is a plain average over the window ($1/9 \approx 0.111$).

In [ ]:
sanitizer_history = [1, 0, 0, 0, 0, 0, 0, 0, 0]  # lag 1, lag 2, ..., lag 9

for eta in (0.5, 0.9, 1.0):
    idx = exposure_index(sanitizer_history, eta=eta)
    print(f"exposure_index(recent-only purchase, eta={eta}) -> {idx:.4f}")

## 2. Inventory stock and bins

Lotion inventory is a latent stock $\xi^*_t$ that depreciates each week and
increases by one unit whenever lotion is bought ($d_t = 1$):

$$
\xi^*_{t+1} = \rho\, \xi^*_t + d_t, \qquad 0 < \rho < 1 .
$$

The continuous stock is then mapped to a discrete bin:
`"0"` for $[0, 0.5)$, `"1"` for $[0.5, 1.5)$, `"2"` for $[1.5, 2.5)$, and
`"3+"` for $[2.5, \infty)$.

The purchase schedule below reproduces the worked example in the underlying
paper with $\rho = 0.84$: purchases in weeks 1 and 4, starting from an empty
stock.

In [ ]:
rho = 0.84
purchase_schedule = [True, False, False, True, False]

stock = 0.0
for week, purchased in enumerate(purchase_schedule, start=1):
    stock = inventory_transition(stock, purchased=purchased, rho=rho)
    stock_bin = bin_inventory(stock)
    print(f"week {week}: d={str(purchased):<5} stock={stock:.3f} bin={stock_bin}")

## 3. Synthetic household-week panel

`generate_synthetic_panel` simulates households week by week. Sanitizer
purchases are random, and each week's lotion-purchase probability comes from a
known logistic rule built from the two state variables above:

$$
\Pr(\text{buy lotion}_t) = \frac{1}{1 + e^{-(\alpha E_t - \lambda\, \mathrm{bin}(\xi^*_t))}} ,
$$

with the exposure index computed at $\eta = 0.7$. The function returns the
panel **and** the `truth` dictionary of generating parameters, so later
model-fitting steps can be checked against known values.

The columns match the real panel's schema; `san_lag1`–`san_lag9` are the
sanitizer lags fed into the exposure index, and `lot_lag1` is last week's
lotion purchase.

In [ ]:
panel, truth = generate_synthetic_panel(n_households=5, n_weeks=12, seed=42)

n_h, n_w = truth["n_households"], truth["n_weeks"]
print(f"Generated {len(panel)} rows from {n_h} households x {n_w} weeks")
truth

Weekly record of the first household:

In [ ]:
first_household = panel[panel["household_code"] == panel["household_code"].iloc[0]]
first_household[["week", "bought_sanitizer", "bought_lotion", "san_lag1", "lot_lag1"]]

## 4. Forward simulation and multipliers

`simulate_response` follows 500 households for 26 weeks under two scenarios
that share the same random draws: a **baseline**, and a **shocked** scenario in
which every household also buys sanitizer in week 0. Lotion purchases follow
the same logistic rule as the synthetic panel. The **response** is the change
in the average weekly lotion-purchase probability caused by the shock.

The shock reaches the exposure index one week later, so the response is zero
in week 0, jumps in week 1, and fades. Households that bought lotion early then
hold more inventory, so later weeks turn slightly **negative**: the extra
purchases are partly offset.

Two multipliers summarize the path (expected lotion purchases per household):

- **gross** — sum of the positive weekly responses (extra purchases only);
- **net** — sum of all weekly responses, after the later offset.

In [ ]:
result = simulate_response(n_weeks=26, n_households=500, seed=0)
result.head(12).round(4)

In [ ]:
response_multipliers(result["response"])

## Next steps

- `scripts/generate_example_data.py` writes the larger committed example
  dataset (`data/example/synthetic_panel.csv`, 50 households x 20 weeks) from
  a fixed seed.
- The decision rule above uses fixed coefficients (`alpha = 1.5`, `lam = 0.6`).
  Estimating them from data (the data-driven layer) is the next step; see
  `README.md`, *Next Step*.